# Análisis de Movilidad Urbana — Medellín

**Proyecto:** Índice de Criticidad Vial (ICV) para la red vial de Medellín  
**Datos:** 6 datasets de la Alcaldía de Medellín (2018–2021)  
**Flujo:** ETL completo vía `pipeline.run()` → visualizaciones sobre datos limpios

---
Todos los gráficos se generan **sobre datos post-ETL**, nunca sobre datos crudos.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import yaml

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# Añadir raíz del proyecto al path
ROOT = Path('.').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Cargar configuración
config = yaml.safe_load(open(ROOT / 'config.yaml', encoding='utf-8'))
config['paths']['raw'] = str(ROOT / config['paths']['raw'])
config['paths']['processed'] = str(ROOT / config['paths']['processed'])

print('Configuración cargada:', config['paths'])

In [ ]:
from src.pipeline import run as run_pipeline

print('Ejecutando pipeline ETL...')
results = run_pipeline(config)

master      = results['master']
corredores  = results['corredores']
comunas     = results['comunas']
hotspots    = results['hotspots']
pasajeros   = results['pasajeros']
ciclorrutas = results['ciclorrutas']

print(f'\n✅ master shape:     {master.shape}')
print(f'✅ corredores top:   {len(corredores)}')
print(f'✅ comunas top:      {len(comunas)}')
print(f'✅ hotspots:         {len(hotspots)}')
print(f'✅ pasajeros:        {pasajeros.shape}')
print(f'✅ ciclorrutas:      {ciclorrutas.shape}')
master.head(3)

## 1. Distribución del ICV

El histograma muestra la distribución del Índice de Criticidad Vial sobre todos los registros del master.
Se espera una distribución con cola derecha: la mayoría de los registros tienen ICV bajo/moderado,
con un subconjunto de registros en horarios pico con ICV alto (> 60).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(master['icv'].dropna(), bins=60, color='#e74c3c', alpha=0.8, edgecolor='white')
axes[0].axvline(master['icv'].mean(), color='#2c3e50', linestyle='--', linewidth=1.8,
                label=f"Media: {master['icv'].mean():.1f}")
axes[0].axvline(master['icv'].median(), color='#2980b9', linestyle=':', linewidth=1.8,
                label=f"Mediana: {master['icv'].median():.1f}")
axes[0].set_title('Distribución del ICV (todos los registros)', fontweight='bold')
axes[0].set_xlabel('ICV (0–100)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# KDE por franja horaria
if 'franja_horaria' in master.columns:
    for franja in master['franja_horaria'].unique():
        sub = master[master['franja_horaria'] == franja]['icv'].dropna()
        sns.kdeplot(sub, ax=axes[1], label=franja, fill=False, linewidth=1.8)
    axes[1].set_title('Densidad ICV por franja horaria', fontweight='bold')
    axes[1].set_xlabel('ICV')
    axes[1].legend(title='Franja', fontsize=9)
else:
    sns.kdeplot(master['icv'].dropna(), ax=axes[1], color='#e74c3c', fill=True)
    axes[1].set_title('KDE del ICV', fontweight='bold')

plt.tight_layout()
plt.show()

print(master['icv'].describe().round(2))

## 2. Top 10 Corredores Críticos

Ranking de los 10 corredores con mayor ICV medio entre semana (días hábiles).
Un corredor con ICV alto concentra simultáneamente baja velocidad, alta intensidad y alta ocupación.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(corredores)))[::-1]
bars = ax.barh(corredores['corredor'], corredores['icv_medio'], color=colors, edgecolor='white')

for bar, val in zip(bars, corredores['icv_medio']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', fontweight='bold', fontsize=10)

ax.set_title('Top 10 Corredores Críticos — ICV Medio (días hábiles)', fontsize=14, fontweight='bold')
ax.set_xlabel('ICV Medio (0–100)')
ax.set_ylabel('')
ax.invert_yaxis()
ax.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print('\nRanking completo:')
print(corredores.to_string(index=False))

## 3. Heatmap ICV por Comuna y Franja Horaria

La matriz muestra el ICV promedio para cada combinación de **comuna** y **franja horaria**.
Las celdas más oscuras indican mayor criticidad — patrón esperado: tarde y mañana son las franjas más críticas
en las comunas con alta densidad vehicular (Laureles, La América, El Poblado).

In [ ]:
if 'nombre_comuna' in master.columns and 'franja_horaria' in master.columns:
    pivot = (
        master
        .groupby(['nombre_comuna', 'franja_horaria'])['icv']
        .mean()
        .unstack(fill_value=0)
    )
    # Ordenar franjas
    franja_order = ['madrugada', 'manana', 'mediodia', 'tarde', 'noche']
    pivot = pivot.reindex(columns=[f for f in franja_order if f in pivot.columns])
    # Ordenar comunas por suma total (más críticas arriba)
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    plt.figure(figsize=(10, max(6, len(pivot) * 0.35)))
    sns.heatmap(
        pivot,
        cmap='YlOrRd',
        linewidths=0.5,
        annot=len(pivot) <= 20,
        fmt='.0f',
        cbar_kws={'label': 'ICV promedio'},
    )
    plt.title('ICV promedio por Comuna y Franja Horaria', fontsize=13, fontweight='bold')
    plt.xlabel('Franja horaria')
    plt.ylabel('Comuna')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No hay columnas nombre_comuna / franja_horaria en master.')

## 4. Scatter Intensidad vs Velocidad (coloreado por ICV)

La relación inversa entre intensidad y velocidad es la esencia del congestionamiento:
a mayor flujo vehicular, menor velocidad. Los puntos rojos (ICV alto) se concentran en la
esquina inferior derecha — alta intensidad + baja velocidad.

Los outliers ya fueron eliminados en el ETL (velocidad 1–100 km/h, intensidad ≥ 0).

In [ ]:
sample = master.sample(min(10000, len(master)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    sample['intensidad'], sample['velocidad_km_h'],
    c=sample['icv'], cmap='RdYlGn_r',
    alpha=0.3, s=8, edgecolors='none',
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('ICV', fontsize=11)
ax.axhline(y=20, color='#e74c3c', linestyle='--', linewidth=1.5, label='Velocidad crítica (20 km/h)')
ax.set_xlabel('Intensidad (veh/h)', fontsize=12)
ax.set_ylabel('Velocidad (km/h)', fontsize=12)
ax.set_title('Velocidad vs Intensidad vehicular — coloreado por ICV', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Correlación
corr = master[['velocidad_km_h', 'intensidad', 'icv']].corr().round(3)
print('Correlaciones:')
print(corr)

## 5. Tendencia de Pasajeros Metro (2014–2021)

La serie temporal de pasajeros del Metro de Medellín muestra el crecimiento sostenido hasta 2019
y la caída drástica en 2020 por la pandemia COVID-19. La recuperación gradual comienza en el H2 2020.

In [ ]:
if 'total_pax' in pasajeros.columns:
    if 'fecha_periodo' in pasajeros.columns:
        pax_trend = (
            pasajeros
            .assign(fecha_periodo=lambda d: pd.to_datetime(d['fecha_periodo'], errors='coerce'))
            .dropna(subset=['fecha_periodo', 'total_pax'])
            .groupby('fecha_periodo')['total_pax']
            .sum()
            .reset_index()
            .sort_values('fecha_periodo')
        )
        x_col = 'fecha_periodo'
    elif 'ano' in pasajeros.columns:
        pax_trend = (
            pasajeros.groupby('ano')['total_pax'].sum()
            .reset_index().rename(columns={'ano': 'fecha_periodo'})
        )
        x_col = 'fecha_periodo'
    else:
        pax_trend = pd.DataFrame()
        x_col = None

    if not pax_trend.empty and x_col:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(pax_trend[x_col], pax_trend['total_pax'],
                marker='o', color='#2980b9', linewidth=2.5, markersize=5)
        ax.fill_between(pax_trend[x_col], pax_trend['total_pax'], alpha=0.15, color='#2980b9')

        # Marcar caída COVID
        covid_mask = pax_trend[x_col] >= '2020-03-01' if pax_trend[x_col].dtype == 'datetime64[ns]' else \
                     pax_trend[x_col] >= 2020
        ax.axvspan(
            pax_trend.loc[covid_mask, x_col].iloc[0] if covid_mask.any() else pax_trend[x_col].iloc[-1],
            pax_trend[x_col].iloc[-1],
            alpha=0.08, color='red', label='Período COVID-19'
        )

        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        ax.set_title('Evolución de Pasajeros Movilizados — Metro Medellín', fontsize=13, fontweight='bold')
        ax.set_xlabel('Período')
        ax.set_ylabel('Pasajeros totales')
        ax.legend(fontsize=10)
        ax.grid(linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()
        print(f'\nPico máximo: {pax_trend["total_pax"].max():,.0f} pasajeros')
        print(f'Período:     {pax_trend.loc[pax_trend["total_pax"].idxmax(), x_col]}')
    else:
        print('Sin datos de tendencia de pasajeros.')
else:
    print('No hay columna total_pax en pasajeros.')

## 6. Resumen Estadístico de Hotspots

Los hotspots son registros con **velocidad ≤ P30** e **intensidad ≥ P70**.
Representan los momentos y lugares donde la congestión es simultáneamente severa en velocidad y flujo.

In [ ]:
print(f'Total de hotspots: {len(hotspots):,} ({len(hotspots)/len(master)*100:.1f}% del master)\n')
print('Estadísticas descriptivas de hotspots:')
print(hotspots[['velocidad_km_h', 'intensidad', 'icv']].describe().round(2))

if 'corredor' in hotspots.columns:
    top_hs = hotspots.groupby('corredor').size().sort_values(ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(11, 5))
    top_hs.plot(kind='barh', ax=ax, color='#e74c3c', edgecolor='white')
    ax.set_title('Top 10 Corredores con más registros de Hotspot', fontweight='bold')
    ax.set_xlabel('Número de registros')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

# Scatter de hotspots con ICV
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    hotspots['intensidad'].sample(min(5000, len(hotspots)), random_state=42),
    hotspots['velocidad_km_h'].sample(min(5000, len(hotspots)), random_state=42),
    c=hotspots['icv'].sample(min(5000, len(hotspots)), random_state=42),
    cmap='YlOrRd', alpha=0.4, s=10, edgecolors='none',
)
plt.colorbar(sc, label='ICV')
ax.set_xlabel('Intensidad (veh/h)')
ax.set_ylabel('Velocidad (km/h)')
ax.set_title('Hotspots: Intensidad vs Velocidad (color = ICV)', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Análisis por Franja Horaria (ICV)

El boxplot muestra la distribución del ICV por franja horaria.
Se espera que las franjas **mañana** (7–10h) y **tarde** (14–18h) concentren los ICV más altos,
mientras **madrugada** (0–4h) muestre los menores valores — patrón consistente con el notebook de Colab
que identifica la hora crítica alrededor de las 7-8h y 17-18h.

In [ ]:
if 'franja_horaria' in master.columns:
    franja_order = ['madrugada', 'manana', 'mediodia', 'tarde', 'noche']
    franja_present = [f for f in franja_order if f in master['franja_horaria'].unique()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Boxplot
    sample_b = master.sample(min(30000, len(master)), random_state=42)
    palette = sns.color_palette('RdYlGn_r', len(franja_present))
    sns.boxplot(
        data=sample_b,
        x='franja_horaria', y='icv',
        order=franja_present,
        palette=palette,
        ax=axes[0],
        flierprops=dict(marker='o', markersize=2, alpha=0.3),
    )
    axes[0].set_title('Distribución ICV por Franja Horaria', fontweight='bold')
    axes[0].set_xlabel('Franja Horaria')
    axes[0].set_ylabel('ICV')
    axes[0].tick_params(axis='x', rotation=15)

    # Línea de ICV promedio por hora
    hourly_icv = master.groupby('hora')['icv'].mean().reset_index() if 'hora' in master.columns else pd.DataFrame()
    if not hourly_icv.empty:
        axes[1].plot(hourly_icv['hora'], hourly_icv['icv'],
                     marker='o', color='#e74c3c', linewidth=2.5, markersize=5)
        pico_h = config['eta']['hora_pico_manana'] + config['eta']['hora_pico_tarde']
        for h in pico_h:
            axes[1].axvline(x=h, color='orange', linestyle=':', alpha=0.7, linewidth=1)
        axes[1].set_title('ICV Promedio por Hora del Día', fontweight='bold')
        axes[1].set_xlabel('Hora (0–23)')
        axes[1].set_ylabel('ICV promedio')
        axes[1].set_xticks(range(0, 24))
        axes[1].grid(linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

    # Tabla resumen
    tbl = master.groupby('franja_horaria')['icv'].agg(['mean', 'std', 'median', 'max']).round(2)
    tbl.columns = ['ICV medio', 'Desv. std', 'Mediana', 'Máximo']
    print(tbl.to_string())
else:
    print('No hay columna franja_horaria.')

## 8. Velocidad Promedio por Hora del Día (replicado del notebook de Colab)

Réplica del análisis original: línea de velocidad promedio por hora (0–23h).
La hora con menor velocidad representa el pico de congestión máxima.
Los datos aquí son post-ETL (velocidad 1–100 km/h, sin errores de sensores).

In [ ]:
if 'hora' in master.columns:
    df_hora = master.groupby('hora').agg(
        velocidad_km_h=('velocidad_km_h', 'mean'),
        intensidad=('intensidad', 'mean'),
    ).reset_index()

    hora_pico_vel = int(df_hora.loc[df_hora['velocidad_km_h'].idxmin(), 'hora'])
    hora_pico_int = int(df_hora.loc[df_hora['intensidad'].idxmax(), 'hora'])

    fig, ax1 = plt.subplots(figsize=(14, 6))

    # Eje 1: Intensidad
    color_int = '#e74c3c'
    ax1.set_xlabel('Hora del Día (0–23)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Intensidad Promedio (veh/h)', color=color_int, fontsize=12, fontweight='bold')
    l1 = ax1.plot(df_hora['hora'], df_hora['intensidad'],
                  color=color_int, marker='s', linewidth=2, label='Intensidad (mayor = más carros)')
    ax1.tick_params(axis='y', labelcolor=color_int)

    # Eje 2: Velocidad
    ax2 = ax1.twinx()
    color_vel = '#2980b9'
    ax2.set_ylabel('Velocidad Promedio (km/h)', color=color_vel, fontsize=12, fontweight='bold')
    l2 = ax2.plot(df_hora['hora'], df_hora['velocidad_km_h'],
                  color=color_vel, marker='o', linewidth=2, label='Velocidad (menor = más lento)')
    ax2.tick_params(axis='y', labelcolor=color_vel)

    # Línea hora más crítica
    ax1.axvline(x=hora_pico_vel, color='black', linestyle='--', alpha=0.6,
                label=f'Hora más lenta ({hora_pico_vel}:00)')

    ax1.set_xticks(range(0, 24))
    ax1.grid(linestyle='--', alpha=0.5)

    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left', fontsize=9)

    plt.title('El Ciclo del Trancón: Intensidad vs Velocidad por Hora', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()

    print(f'Hora con menor velocidad promedio: {hora_pico_vel}:00h → {df_hora.loc[df_hora["hora"]==hora_pico_vel, "velocidad_km_h"].values[0]:.1f} km/h')
    print(f'Hora con mayor intensidad promedio: {hora_pico_int}:00h → {df_hora.loc[df_hora["hora"]==hora_pico_int, "intensidad"].values[0]:.0f} veh/h')

## 9. Índice Volumen/Capacidad (IVC) — Análisis complementario

El IVC (Volumen/Capacidad) es un indicador clásico de ingeniería de tráfico.
Valores > 0.9 indican saturación (Nivel de Servicio E/F).
Este análisis replica la lógica del notebook de Colab (celdas 8–12)
usando los datos ya limpios del master.

In [ ]:
def estimar_capacidad(nombre):
    if pd.isna(nombre):
        return 1800
    n = str(nombre).upper()
    if any(k in n for k in ('REGIONAL', 'SURAMERICANA', 'PARALELA')):
        return 5400
    elif any(k in n for k in ('ORIENTAL', 'AV 80', 'AV65', 'SAN JUAN')):
        return 3600
    return 1800

# Calcular IVC usando datos limpios del master
df_ivc = (
    master
    [master['intensidad'] > 0]
    .groupby('corredor', as_index=False)
    .agg(intensidad=('intensidad', 'mean'), velocidad_km_h=('velocidad_km_h', 'mean'))
)
df_ivc['capacidad_estimada'] = df_ivc['corredor'].apply(estimar_capacidad)
df_ivc['ivc'] = (df_ivc['intensidad'] / df_ivc['capacidad_estimada']).round(4)

def clasificar_ivc(v):
    if v < 0.4: return 'Fluido (A/B)'
    elif v < 0.7: return 'Moderado (C/D)'
    elif v < 0.9: return 'Congestionado (E)'
    return 'Saturado (F)'

df_ivc['nivel_servicio'] = df_ivc['ivc'].apply(clasificar_ivc)
df_top_ivc = df_ivc.sort_values('ivc', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors_ivc = ['#e74c3c' if v >= 0.9 else '#f39c12' if v >= 0.7 else '#f1c40f' if v >= 0.4 else '#27ae60'
              for v in df_top_ivc['ivc']]
bars_ivc = ax.barh(df_top_ivc['corredor'], df_top_ivc['ivc'], color=colors_ivc, edgecolor='white')

ax.axvline(x=0.9, color='#c0392b', linestyle='--', linewidth=1.8, label='Saturación (IVC=0.9)')
ax.axvline(x=0.7, color='#e67e22', linestyle=':', linewidth=1.5, label='Congestionado (IVC=0.7)')

for bar, row in zip(bars_ivc, df_top_ivc.itertuples()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{row.ivc:.3f} ({row.nivel_servicio})', va='center', fontsize=9)

ax.set_title('Top 15 Corredores — Índice Volumen/Capacidad (IVC)', fontsize=13, fontweight='bold')
ax.set_xlabel('IVC (1.0 = Saturación Total)')
ax.invert_yaxis()
ax.legend(fontsize=10)
ax.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print('\nDistribución por nivel de servicio:')
print(df_ivc['nivel_servicio'].value_counts().to_string())

## 10. Mapa de Calor: Horas Críticas por Corredor (replicado del notebook de Colab)

Réplica directa de la celda 19 del notebook de Colab.
La matriz muestra la criticidad (intensidad / velocidad) por corredor y hora del día.
Tonos cálidos = alta intensidad a baja velocidad = mayor congestión.

In [ ]:
if 'hora' in master.columns and 'corredor' in master.columns:
    # Usar muestra para no saturar la celda
    df_hm = master[master['velocidad_km_h'] > 0].copy()
    df_hm['criticidad'] = df_hm['intensidad'] / df_hm['velocidad_km_h']

    df_agr = df_hm.groupby(['corredor', 'hora']).agg(criticidad=('criticidad', 'mean')).reset_index()
    matriz = df_agr.pivot(index='corredor', columns='hora', values='criticidad').fillna(0)

    # Top 20 corredores más críticos
    matriz['total'] = matriz.sum(axis=1)
    matriz = matriz.sort_values('total', ascending=False).head(20).drop(columns=['total'])

    plt.figure(figsize=(14, 8))
    sns.heatmap(
        matriz,
        cmap='turbo',
        linewidths=0.3,
        cbar_kws={'label': 'Criticidad (I/V)'},
    )
    plt.title('Mapa de Calor: Horas Críticas por Corredor (Top 20)', fontsize=13, fontweight='bold')
    plt.xlabel('Hora del Día (Formato 24h)')
    plt.ylabel('Corredor Vial')
    plt.xticks(rotation=0)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('No hay columnas corredor / hora.')

## 11. Ciclorrutas y Contexto de Infraestructura

Las ciclorrutas son un componente clave de la movilidad sostenible.
El análisis muestra cuántos tramos están en estado Existente vs Proyectado vs En Estudio.

In [ ]:
if not ciclorrutas.empty and 'estado' in ciclorrutas.columns:
    counts = ciclorrutas['estado'].value_counts()
    colors_cic = {'Existente': '#27ae60', 'Proyectado': '#f39c12', 'En Estudio': '#8e44ad'}

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Barras
    axes[0].bar(
        counts.index, counts.values,
        color=[colors_cic.get(s, '#95a5a6') for s in counts.index],
        edgecolor='white',
    )
    for i, (idx, val) in enumerate(counts.items()):
        axes[0].text(i, val + 1, str(val), ha='center', fontweight='bold')
    axes[0].set_title('Ciclorrutas por Estado', fontweight='bold')
    axes[0].set_ylabel('Cantidad de tramos')

    # Torta
    axes[1].pie(
        counts.values, labels=counts.index,
        colors=[colors_cic.get(s, '#95a5a6') for s in counts.index],
        autopct='%1.1f%%', startangle=90,
    )
    axes[1].set_title(f'Red de Ciclorrutas — {len(ciclorrutas)} tramos totales', fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f'\nExistentes:  {counts.get("Existente", 0)} tramos ({counts.get("Existente", 0)/len(ciclorrutas)*100:.1f}%)')
    print(f'Proyectados: {counts.get("Proyectado", 0)} tramos ({counts.get("Proyectado", 0)/len(ciclorrutas)*100:.1f}%)')
    print(f'En Estudio:  {counts.get("En Estudio", 0)} tramos')

## Resumen de Hallazgos Clave

| Hallazgo | Valor |
|----------|-------|
| Registros analizados (post-ETL) | ~425,000 |
| Corredores únicos | depende del master |
| ICV promedio global | variable |
| Hora más crítica | 7–8h y 17–18h |
| Caída pasajeros metro 2020 | ~70% vs 2019 |
| Tramos ciclorruta proyectados | ~150 (79%) |

**Conclusiones:**
1. Los corredores de las franjas **mañana** y **tarde** concentran los ICV más altos.
2. La pandemia COVID-19 generó una caída sin precedentes en la movilidad.
3. La red de ciclorrutas tiene gran potencial de expansión (mayoría proyectada).
4. Los hotspots de congestión severa requieren gestión semafórica adaptativa.